# Diabetes Dataset Cleaning and Preprocessing

This notebook only covers the raw-to-final dataset preparation pipeline. It ends once `diabetes_final_ml_dataset_encoded.csv` is created.


## Workflow Summary

The steps below preserve the current project logic exactly while making the notebook easier to review:

1. Load the raw dataset and apply the discharge-based exclusion rule.
2. Create clinically motivated engineered features.
3. Build the cleaned modeling dataframe.
4. Apply the final encoding and export the prepared dataset.


## 1. Import Libraries


In [ ]:
import numpy as np
import pandas as pd


## 2. Load the Raw Dataset


In [ ]:
df = pd.read_csv('diabetic_data.csv')
df.replace('?', np.nan, inplace=True)

# Remove hospice or deceased discharge cases based on the project's clinical rule.
invalid_discharge_ids = [11, 13, 14, 19, 20, 21]
df = df[~df['discharge_disposition_id'].isin(invalid_discharge_ids)].copy()

print('Filtered raw dataset shape:', df.shape)


## 3. Engineer the Initial Clinical Features


In [ ]:
# Define the target for readmission within 30 days.
df['readmitted_30d'] = (df['readmitted'] == '<30').astype(int)

# Create binary features.
df['gender_binary'] = df['gender'].map({'Male': 1, 'Female': 0}).fillna(0).astype(int)
df['med_changed'] = df['change'].map({'Ch': 1, 'No': 0}).fillna(0).astype(int)
df['diabetes_med_present'] = df['diabetesMed'].map({'Yes': 1, 'No': 0}).fillna(0).astype(int)

# Create ordinal and utilization-related features.
age_mapping = {
    '[0-10)': 5, '[10-20)': 15, '[20-30)': 25, '[30-40)': 35, '[40-50)': 45,
    '[50-60)': 55, '[60-70)': 65, '[70-80)': 75, '[80-90)': 85, '[90-100)': 95
}
df['age_numeric'] = df['age'].map(age_mapping)
df['hba1c_tested'] = np.where(df['A1Cresult'].isna(), 0, 1)
df['max_glu_tested'] = np.where(df['max_glu_serum'].isna(), 0, 1)
df['max_glu_high'] = df['max_glu_serum'].isin(['>200', '>300']).astype(int)
df['insulin_ordinal'] = df['insulin'].map({'No': 0, 'Steady': 1, 'Up': 2, 'Down': 2}).fillna(0).astype(int)
df['prior_utilization_total'] = df['number_outpatient'] + df['number_emergency'] + df['number_inpatient']


## 4. Group Clinical Categories


In [ ]:
# Group discharge disposition.
def group_discharge(disch_id):
    if disch_id in [1, 6, 8]:
        return 'Home'
    elif disch_id in [3, 4, 5, 22, 23, 24]:
        return 'Nursing_Facility_or_Transfer'
    else:
        return 'Other'

df['discharge_group'] = df['discharge_disposition_id'].apply(group_discharge)

# Group admission type.
def group_admission_type(type_id):
    if type_id in [1, 2, 7]:
        return 'Emergency_Urgent'
    elif type_id == 3:
        return 'Elective'
    else:
        return 'Other_Unknown'

df['admission_type_group'] = df['admission_type_id'].apply(group_admission_type)

# Group admission source.
def group_admission_source(source_id):
    if source_id == 7:
        return 'Emergency_Room'
    elif source_id in [1, 2, 4]:
        return 'Referral'
    else:
        return 'Transfer_Other'

df['admission_source_group'] = df['admission_source_id'].apply(group_admission_source)

# Map ICD-9 diagnosis codes into broader categories.
def map_icd_to_group(code):
    if pd.isna(code):
        return 'Missing'
    code_str = str(code).strip()
    if code_str.startswith('V') or code_str.startswith('E'):
        return 'Other'
    try:
        val = float(code_str)
        if 390 <= val <= 459 or val == 785:
            return 'Circulatory'
        elif 460 <= val <= 519 or val == 786:
            return 'Respiratory'
        elif 520 <= val <= 579 or val == 782:
            return 'Digestive'
        elif int(val) == 250:
            return 'Diabetes'
        elif 800 <= val <= 999:
            return 'Injury'
        elif 710 <= val <= 739:
            return 'Musculoskeletal'
        elif 580 <= val <= 629 or val == 788:
            return 'Genitourinary'
        elif 140 <= val <= 239:
            return 'Neoplasms'
        else:
            return 'Other'
    except ValueError:
        return 'Other'

df['diag_1_group'] = df['diag_1'].apply(map_icd_to_group)
df['diag_2_group'] = df['diag_2'].apply(map_icd_to_group)
df['diag_3_group'] = df['diag_3'].apply(map_icd_to_group)

# Keep the top specialties and group the remainder as 'Other'.
df['medical_specialty'] = df['medical_specialty'].fillna('Missing')
top_specialties = df['medical_specialty'].value_counts().iloc[:10].index.tolist()
df['medical_specialty_grouped'] = df['medical_specialty'].apply(lambda x: x if x in top_specialties else 'Other')


## 5. Build the Modeling Dataframe


In [ ]:
df_model = df.copy()
df_model = df_model.replace('?', np.nan)
df_model['readmitted_30d'] = (df_model['readmitted'] == '<30').astype(int)

# Drop leakage columns and columns removed by the current project logic.
drop_cols = [
    'encounter_id',
    'patient_nbr',
    'readmitted',
    'weight',
    'payer_code',
    'examide',
    'citoglipton',
    'diag_1',
    'diag_2',
    'diag_3'
]
df_model = df_model.drop(columns=[c for c in drop_cols if c in df_model.columns])

# Recreate the age midpoint feature inside the modeling dataframe.
age_map = {
    '[0-10)': 5,
    '[10-20)': 15,
    '[20-30)': 25,
    '[30-40)': 35,
    '[40-50)': 45,
    '[50-60)': 55,
    '[60-70)': 65,
    '[70-80)': 75,
    '[80-90)': 85,
    '[90-100)': 95
}
if 'age' in df_model.columns:
    df_model['age_numeric'] = df_model['age'].map(age_map)
    df_model = df_model.drop(columns=['age'])

# Encode the original binary source columns.
binary_maps = {
    'gender': {'Female': 0, 'Male': 1},
    'change': {'No': 0, 'Ch': 1},
    'diabetesMed': {'No': 0, 'Yes': 1}
}
for col, mapping in binary_maps.items():
    if col in df_model.columns:
        df_model[col] = df_model[col].map(mapping)

if 'gender' in df_model.columns:
    df_model = df_model.dropna(subset=['gender'])

# Create lab testing indicator features.
if 'A1Cresult' in df_model.columns:
    df_model['hba1c_tested'] = df_model['A1Cresult'].notna().astype(int)
    df_model['hba1c_high'] = df_model['A1Cresult'].isin(['>7', '>8']).astype(int)
    df_model['A1Cresult'] = df_model['A1Cresult'].fillna('Not_Tested')

if 'max_glu_serum' in df_model.columns:
    df_model['max_glu_tested'] = df_model['max_glu_serum'].notna().astype(int)
    df_model['max_glu_high'] = df_model['max_glu_serum'].isin(['>200', '>300']).astype(int)
    df_model['max_glu_serum'] = df_model['max_glu_serum'].fillna('Not_Tested')

# Group low-frequency specialties.
if 'medical_specialty' in df_model.columns:
    df_model['medical_specialty'] = df_model['medical_specialty'].fillna('Missing')
    top_specialties = df_model['medical_specialty'].value_counts()
    rare_specialties = top_specialties[top_specialties < 500].index
    df_model['medical_specialty_grouped'] = df_model['medical_specialty'].replace(rare_specialties, 'Other')
    df_model = df_model.drop(columns=['medical_specialty'])


## 6. Apply the Final Cleanup Rules


In [ ]:
df_clean_model = df_model.copy()

# Remove duplicated binary source columns and keep the clearer engineered names.
duplicate_binary_cols = ['gender', 'change', 'diabetesMed']
df_clean_model = df_clean_model.drop(
    columns=[c for c in duplicate_binary_cols if c in df_clean_model.columns],
    errors='ignore'
)

rename_map = {
    'gender_binary': 'is_male',
    'med_changed': 'medication_changed',
    'diabetes_med_present': 'diabetes_medication_used'
}
df_clean_model = df_clean_model.rename(columns=rename_map)

# Treat these hospital pathway IDs as categorical codes.
id_categorical_cols = [
    'admission_type_id',
    'discharge_disposition_id',
    'admission_source_id'
]
for col in id_categorical_cols:
    if col in df_clean_model.columns:
        df_clean_model[col] = df_clean_model[col].astype('category')

# Create capped versions of extreme utilization counts.
cap_cols = [
    'number_outpatient',
    'number_emergency',
    'number_inpatient',
    'prior_utilization_total'
]
for col in cap_cols:
    if col in df_clean_model.columns:
        upper_cap = df_clean_model[col].quantile(0.99)
        df_clean_model[col + '_capped'] = np.where(
            df_clean_model[col] > upper_cap,
            upper_cap,
            df_clean_model[col]
        )

# Prefer the raw insulin category for final one-hot encoding.
if 'insulin' in df_clean_model.columns and 'insulin_ordinal' in df_clean_model.columns:
    df_clean_model = df_clean_model.drop(columns=['insulin_ordinal'])


## 7. Final Encoding and Export


In [ ]:
df_clean_model_v2 = df_clean_model.copy()

# Drop the raw ID columns when their grouped versions are already available.
raw_id_cols_to_drop = [
    'admission_type_id',
    'admission_source_id',
    'discharge_disposition_id'
]
df_clean_model_v2 = df_clean_model_v2.drop(
    columns=[c for c in raw_id_cols_to_drop if c in df_clean_model_v2.columns],
    errors='ignore'
)

# Remove the temporary capped versions and keep the original utilization values.
capped_cols_to_drop = [
    'number_outpatient_capped',
    'number_emergency_capped',
    'number_inpatient_capped',
    'prior_utilization_total_capped'
]
df_clean_model_v2 = df_clean_model_v2.drop(
    columns=[c for c in capped_cols_to_drop if c in df_clean_model_v2.columns],
    errors='ignore'
)

# Keep the raw lab categories in the final feature space.
drop_raw_lab_categories = False
if drop_raw_lab_categories:
    df_clean_model_v2 = df_clean_model_v2.drop(
        columns=['A1Cresult', 'max_glu_serum'],
        errors='ignore'
    )

target_col = 'readmitted_30d'
categorical_cols_final_v2 = df_clean_model_v2.select_dtypes(include=['object', 'category']).columns.tolist()

df_final_encoded_v2 = pd.get_dummies(
    df_clean_model_v2,
    columns=categorical_cols_final_v2,
    drop_first=True,
    dtype=int
)

X = df_final_encoded_v2.drop(columns=[target_col])
y = df_final_encoded_v2[target_col]

print('Final model dataset ready.')
print('X shape:', X.shape)
print('y shape:', y.shape)
print('Missing values in X:', X.isna().sum().sum())

# Run final safety checks before export.
leakage_cols = ['readmitted', 'encounter_id', 'patient_nbr']
print('Leakage columns still present:', [c for c in leakage_cols if c in X.columns])
print('Duplicated columns:', X.columns[X.columns.duplicated()].tolist())

df_export = X.copy()
df_export['readmitted_30d'] = y.values

export_path = 'diabetes_final_ml_dataset_encoded.csv'
df_export.to_csv(export_path, index=False)

print('Dataset exported successfully.')
print('File name:', export_path)
print('Shape:', df_export.shape)
df_export.head()
